#Week 7: Retrieval-Augmented Generation (RAG) System using Custom PDF



## Retrieval-Augmented Generation (RAG) System using Custom Documents

### Objective
To build a simple Retrieval-Augmented Generation (RAG) pipeline that retrieves relevant information from a custom document and generates answers to user queries.

### Document Used
Internet of Things: A Survey on Enabling Technologies, Protocols, and Applications

In [1]:
!pip install sentence-transformers faiss-cpu PyPDF2 transformers torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 11.3 MB/s eta 0:00:00


## Import Required Libraries

In [2]:
import PyPDF2
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import pipeline

## Load and Extract Text from PDF

In [4]:
pdf_path = "/content/Internet_of_Things_A_Survey_on_Enabling_Technologies_Protocols_and_Applications.pdf"

text = ""

with open(pdf_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)

    for page in reader.pages:
        text += page.extract_text()

print("Total Characters:", len(text))

Total Characters: 177242


## Split Document into Chunks

In [5]:
chunk_size = 500

chunks = []

for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i+chunk_size])

print("Total Chunks:", len(chunks))

Total Chunks: 355


## Create Embeddings using Sentence Transformers

In [6]:
embedding_model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

embeddings = embedding_model.encode(chunks)

print("Embedding Shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Shape: (355, 384)


## Store Embeddings in FAISS Vector Database

In [7]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Vectors Stored:", index.ntotal)

Vectors Stored: 355


## Retrieve Relevant Context

In [8]:
def retrieve(query, top_k=3):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        top_k
    )

    retrieved_chunks = []

    for idx in indices[0]:
        retrieved_chunks.append(chunks[idx])

    return retrieved_chunks

## Load Text Generation Model

In [9]:
generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## Build RAG Question Answering Pipeline

In [10]:
def answer_question(question):

    retrieved = retrieve(question)

    context = " ".join(retrieved)

    prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

    result = generator(
        prompt,
        max_new_tokens=100,
        do_sample=False
    )

    return retrieved, result[0]['generated_text']

## Testing the RAG System

In [11]:
retrieved, answer = answer_question(
    "What is Internet of Things?"
)

print(answer)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_co


Context:
012, pp. 257–260.
[4] J. Gubbi, R. Buyya, S. Marusic, and M. Palaniswami, “Internet of Things
(IoT): A vision, architectural elements, and future directions,” Future
Gener. Comput. Syst. , vol. 29, no. 7, pp. 1645–1660, Sep. 2013.
[5] P. Lopez, D. Fernandez, A. J. Jara, and A. F. Skarmeta, “Survey of
Internet of Things technologies for clinical environments,” in Proc. 27th
Int. Conf. WAINA , 2013, pp. 1349–1354.
[6] D. Yang, F. Liu, and Y . Liang, “A survey of the Internet of Things,” in
Proc. 1 ) integrated with sensors and
built-in TCP/IP and security functionalities are typically used
to realize IoT products (e.g., Arduino Yun, Raspberry PI, Bea-
gleBone Black, etc.). Such devices typically connect to a centralmanagement portal to provide the required data by customers.
C. Communication
The IoT communication technologies connect heterogeneous
objects together to deliver speciﬁc smart services. Typically, the
IoT nodes should operate using low power in the presence of
lossy

In [12]:
retrieved, answer = answer_question(
    "What is MQTT protocol?"
)

print(answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Context:
and simplicity of implementation as depicted in
Fig. 8. Also, MQTT is suitable for resource constrained devices
that use unreliable or low bandwidth links. MQTT is built on
top of the TCP protocol. It delivers messages through three lev-els of QoS. Two major speciﬁcations exist for MQTT: MQTT
v3.1 and MQTT-SN [71] (formerly known as MQTT-S) V1.2.
The latter was deﬁned speciﬁcally for sensor networks and de-ﬁnes a UDP mapping of MQTT and adds broker support for in-
dexing topic names. The speciﬁc e information to the interested entities (subscribers)
through the broker. Furthermore, the broker achieves securityby checking authorization of the publishers and the subscribers
[71]. Numerous applications utilize the MQTT such as health
care, monitoring, energy meter, and Facebook notiﬁcation.Therefore, the MQTT protocol r epresents an ideal messaging
protocol for the IoT and M2M communications and is able to
provide routing for small, cheap, low power and low memory
devices in vul

In [18]:
print("\nRetrieved Context:\n")
for chunk in retrieved:
    print(chunk[:300])
    print("-"*50)


Retrieved Context:

and simplicity of implementation as depicted in
Fig. 8. Also, MQTT is suitable for resource constrained devices
that use unreliable or low bandwidth links. MQTT is built on
top of the TCP protocol. It delivers messages through three lev-els of QoS. Two major speciﬁcations exist for MQTT: MQTT
v3.1 a
--------------------------------------------------
e information to the interested entities (subscribers)
through the broker. Furthermore, the broker achieves securityby checking authorization of the publishers and the subscribers
[71]. Numerous applications utilize the MQTT such as health
care, monitoring, energy meter, and Facebook notiﬁcation.The
--------------------------------------------------
developerWor ks, Markham, ON, Canada, Tech.
Lib., 2010. [Online]. Available: Http://Www.Ibm.Com/Developerworks/Webservices/Library/Ws-Mqtt/Index.Html
[71] U. Hunkeler, H. L. Truong, and A. Stanford-Clark, “MQTT-S—A
publish/subscribe protocol for wireless sensor networks,” in

In [13]:
retrieved, answer = answer_question(
    "What are the layers of IoT architecture?"
)

print(answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Context:
ver, the number of
layers and their scopes are deﬁned differently depending on theunderlying infrastructures an d technologies. As scalability and
interoperability have a great importance in IoT applications,
augmenting the architecture with better abstractions can ease
these issues. The ﬁve-layer architecture [3], [17], [18] present
such a model.
Then, we introduced the different IoT elements and their
related technologies and tools that are needed to realize an IoT
solution. Identiﬁcation and  e. The ever increas-
ing number of proposed architectures has not yet converged to a
reference model [15]. Meanwhile, there are some projects like
IoT-A [16] which try to design a common architecture based onthe analysis of the needs of researchers and the industry.
From the pool of proposed models, the basic model is a
3-layer architecture [3], [17], [18] consisting of the Applica-
tion, Network, and Perception Layers. In the recent literature,
however, some other models have been pr

In [14]:
retrieved, answer = answer_question(
    "What is CoAP?"
)

print(answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Context:
CoAP include
[65], [68]:
•Resource observation : On-demand subscriptions to
monitor resources of interest using publish/subscribemechanism.
•Block-wise resource transport : Ability to exchange
transceiver data between the client and the server without
Authorized licensed use limited to: Indian Institute Of Technology Jammu. Downloaded on June 16,2026 at 06:53:37 UTC from IEEE Xplore.  Restrictions apply. 2354 IEEE COMMUNICATION SURVEYS & TUTORIALS, VOL. 17, NO. 4, FOURTH QUARTER 2015
Fig. 6. CoA e client. In CoAP’s non-conﬁrmable
response mode, the client sends data without waiting for an
ACK message, while message IDs are used to detect duplicates.
The server side responds with a RST message when messagesare missed or communication issues occur. CoAP, as in HTTP,
utilizes methods such as GET, PUT, POST and DELETE to
achieve Create, Retrieve, Update and Delete (CRUD) opera-tions. For example, the GET method can be used by a server to
inquire the client’s temperature using the

In [15]:
retrieved, answer = answer_question(
    "What are the applications of IoT?"
)

print(answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Context:
ytime they are needed to any-
onewho needs them anywhere . With this categorization, we re-
view some applications of the IoT in the following paragraphs.
The ultimate goal of all IoT applications is to reach the level of
ubiquitous services. However, this end is not achievable easily
since there are a lot of difﬁculties and challenges that have to beaddressed. Most of the existing applications provide identity-
related, information aggregation, and collaborative-aware ser-
vices. Smart healthca  model for IoT applications.
IV . IoT E
LEMENTS
Understanding the IoT building blocks helps to gain a better
insight into the real meaning and functionality of the IoT. Inthe following sections we discuss six main elements needed
to deliver the functionality of the IoT as illustrated in Fig. 4.
Table II shows the categories of these elements and examplesof each category.
A. Identiﬁcation
Identiﬁcation is crucial for the IoT to name and match
services with their demand. Many identiﬁcat

## Retrieved Context Example

In [16]:
question = "What is MQTT protocol?"

retrieved = retrieve(question)

for i, chunk in enumerate(retrieved):
    print(f"\nChunk {i+1}\n")
    print(chunk[:500])


Chunk 1

and simplicity of implementation as depicted in
Fig. 8. Also, MQTT is suitable for resource constrained devices
that use unreliable or low bandwidth links. MQTT is built on
top of the TCP protocol. It delivers messages through three lev-els of QoS. Two major speciﬁcations exist for MQTT: MQTT
v3.1 and MQTT-SN [71] (formerly known as MQTT-S) V1.2.
The latter was deﬁned speciﬁcally for sensor networks and de-ﬁnes a UDP mapping of MQTT and adds broker support for in-
dexing topic names. The speciﬁc

Chunk 2

e information to the interested entities (subscribers)
through the broker. Furthermore, the broker achieves securityby checking authorization of the publishers and the subscribers
[71]. Numerous applications utilize the MQTT such as health
care, monitoring, energy meter, and Facebook notiﬁcation.Therefore, the MQTT protocol r epresents an ideal messaging
protocol for the IoT and M2M communications and is able to
provide routing for small, cheap, low power and low memory
devi

## Evaluation Results

The RAG system was tested using five domain-specific questions related to the IoT research paper.

| Question | Retrieval Quality | Answer Quality |
|-----------|------------------|----------------|
| What is Internet of Things? | Relevant | Correct |
| What is MQTT protocol? | Relevant | Correct |
| What are the layers of IoT architecture? | Relevant | Correct |
| What is CoAP? | Relevant | Correct |
| What are the applications of IoT? | Relevant | Correct |

## Observations

1. The PDF was successfully converted into text and divided into smaller chunks.

2. SentenceTransformer embeddings captured semantic meaning from the document.

3. FAISS enabled fast retrieval of the most relevant document sections.

4. The RAG pipeline generated answers based on retrieved context instead of relying only on the language model.

5. Questions related to IoT architecture, MQTT, CoAP, and applications were answered correctly using information retrieved from the document.

6. Retrieved context inspection confirmed that the system selected relevant chunks before generating answers.

## Experiment: Effect of Retrieval Depth (Top-K)

To evaluate retrieval quality, the system was tested using different numbers of retrieved document chunks (Top-K values).

The objective was to observe how the amount of retrieved context affects answer quality and relevance.

In [17]:
question = "What is MQTT protocol?"

for k in [1, 3, 5]:

    retrieved = retrieve(question, top_k=k)

    context = " ".join(retrieved)

    print(f"\n===== Top-K = {k} =====\n")
    print(context[:500])


===== Top-K = 1 =====

and simplicity of implementation as depicted in
Fig. 8. Also, MQTT is suitable for resource constrained devices
that use unreliable or low bandwidth links. MQTT is built on
top of the TCP protocol. It delivers messages through three lev-els of QoS. Two major speciﬁcations exist for MQTT: MQTT
v3.1 and MQTT-SN [71] (formerly known as MQTT-S) V1.2.
The latter was deﬁned speciﬁcally for sensor networks and de-ﬁnes a UDP mapping of MQTT and adds broker support for in-
dexing topic names. The speciﬁc

===== Top-K = 3 =====

and simplicity of implementation as depicted in
Fig. 8. Also, MQTT is suitable for resource constrained devices
that use unreliable or low bandwidth links. MQTT is built on
top of the TCP protocol. It delivers messages through three lev-els of QoS. Two major speciﬁcations exist for MQTT: MQTT
v3.1 and MQTT-SN [71] (formerly known as MQTT-S) V1.2.
The latter was deﬁned speciﬁcally for sensor networks and de-ﬁnes a UDP mapping of MQTT and adds broke

## Retrieval Evaluation

| Top-K | Observation |
|--------|------------|
| 1 | Most focused context but may miss supporting information |
| 3 | Balanced retrieval with relevant supporting details |
| 5 | More information retrieved but may introduce irrelevant content |

### Experiment Observation

The retrieval experiment demonstrated that the number of retrieved chunks significantly affects answer generation quality.

Using Top-K = 1 provided highly focused context but occasionally missed supporting information. Top-K = 3 produced the most balanced results by combining relevance and sufficient context. Top-K = 5 retrieved more information but sometimes introduced unrelated content that could reduce answer precision.

Based on the experiment, Top-K = 3 was selected as the optimal retrieval setting for this project.

## Challenges

1. Large PDF documents require chunking to fit into embedding models.

2. Retrieved chunks may sometimes contain partial information.

3. Small language models such as DistilGPT2 may generate incomplete responses.

4. Retrieval quality depends heavily on chunk size and embedding quality.

## Conclusion

A simple Retrieval-Augmented Generation system was successfully implemented using a custom IoT research paper. The system combined document retrieval using FAISS and semantic embeddings with text generation to answer user questions. The experiment demonstrated how RAG improves response quality by grounding generated answers in external knowledge sources.